# Malware Detection Model Training

This notebook contains the complete pipeline for training the malware detection model.

## 1. Imports and Setup

In [2]:
import os
import json
import logging
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## 2. Data Loading Components

In [3]:
class MalwareDataset(Dataset):
    """
    A memory-efficient PyTorch Dataset for loading malware features.
    """
    def __init__(self, file_paths, labels, api_seq_len=500):
        self.file_paths = file_paths
        self.labels = labels
        self.api_seq_len = api_seq_len

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        paths = self.file_paths[idx]
        label = self.labels[idx]

        api_features = np.load(paths['apis'])
        global_features = np.load(paths['globals'])

        seq_len, num_features = api_features.shape
        if seq_len > self.api_seq_len:
            api_features = api_features[:self.api_seq_len, :]
        elif seq_len < self.api_seq_len:
            padding = np.zeros((self.api_seq_len - seq_len, num_features))
            api_features = np.vstack((api_features, padding))

        api_tensor = torch.from_numpy(api_features).float()
        global_tensor = torch.from_numpy(global_features).float()
        label_tensor = torch.tensor(label, dtype=torch.float32)

        return (api_tensor, global_tensor), label_tensor

## 3. Model Architecture

In [4]:
class GatedCNN(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1):
        super(GatedCNN, self).__init__(),
        self.conv_out = nn.Conv1d(in_channels, out_channels, kernel_size, stride, padding=0)
        self.conv_gate = nn.Conv1d(in_channels, out_channels, kernel_size, stride, padding=0)
        self.sigmoid = nn.Sigmoid()
        self.kernel_size = kernel_size

    def forward(self, x):
        padding_total = self.kernel_size - 1
        padding_left = padding_total // 2
        padding_right = padding_total - padding_left
        x_padded = torch.nn.functional.pad(x, (padding_left, padding_right))
        out = self.conv_out(x_padded)
        gate = self.sigmoid(self.conv_gate(x_padded))
        return out * gate

class MalwareDetectionModel(nn.Module):
    def __init__(self, api_feature_dim=118, global_feature_dim=8, lstm_hidden_dim=100, cnn_filters=128, classifier_hidden_dim=64, dropout_rate=0.5):
        super(MalwareDetectionModel, self).__init__(),
        self.seq_bn1 = nn.BatchNorm1d(api_feature_dim)
        self.gated_cnn2 = GatedCNN(api_feature_dim, cnn_filters, kernel_size=2)
        self.gated_cnn3 = GatedCNN(api_feature_dim, cnn_filters, kernel_size=3)
        self.seq_bn2 = nn.BatchNorm1d(cnn_filters * 2)
        self.bilstm = nn.LSTM(input_size=cnn_filters * 2, hidden_size=lstm_hidden_dim, num_layers=1, bidirectional=True, batch_first=True)
        self.global_mlp = nn.Sequential(nn.Linear(global_feature_dim, 16), nn.ReLU(), nn.BatchNorm1d(16))
        classifier_input_dim = (lstm_hidden_dim * 2) + 16
        self.classifier = nn.Sequential(nn.Linear(classifier_input_dim, classifier_hidden_dim), nn.ReLU(), nn.Dropout(dropout_rate), nn.Linear(classifier_hidden_dim, 1))
        self.final_activation = nn.Sigmoid()

    def forward(self, api_sequence, global_features):
        api_sequence = api_sequence.permute(0, 2, 1)
        seq_out = self.seq_bn1(api_sequence)
        cnn2_out = self.gated_cnn2(seq_out)
        cnn3_out = self.gated_cnn3(seq_out)
        seq_out = torch.cat((cnn2_out, cnn3_out), dim=1)
        seq_out = self.seq_bn2(seq_out)
        seq_out = seq_out.permute(0, 2, 1)
        lstm_out, _ = self.bilstm(seq_out)
        seq_vector = torch.max(lstm_out, dim=1)[0]
        global_vector = self.global_mlp(global_features)
        combined_vector = torch.cat((seq_vector, global_vector), dim=1)
        logits = self.classifier(combined_vector)
        prediction = self.final_activation(logits)
        return prediction.squeeze(1)

## 4. Training and Validation Functions

In [5]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0
    for (api_batch, global_batch), labels_batch in train_loader:
        api_batch, global_batch, labels_batch = api_batch.to(device), global_batch.to(device), labels_batch.to(device)
        optimizer.zero_grad()
        outputs = model(api_batch, global_batch)
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * api_batch.size(0)
        predicted = (outputs > 0.5).float()
        correct_predictions += (predicted == labels_batch).sum().item()
        total_samples += labels_batch.size(0)
    return total_loss / total_samples, correct_predictions / total_samples

def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0
    with torch.no_grad():
        for (api_batch, global_batch), labels_batch in val_loader:
            api_batch, global_batch, labels_batch = api_batch.to(device), global_batch.to(device), labels_batch.to(device)
            outputs = model(api_batch, global_batch)
            loss = criterion(outputs, labels_batch)
            total_loss += loss.item() * api_batch.size(0)
            predicted = (outputs > 0.5).float()
            correct_predictions += (predicted == labels_batch).sum().item()
            total_samples += labels_batch.size(0)
    return total_loss / total_samples, correct_predictions / total_samples

## 5. Main Training Execution

In [ ]:
# --- Configuration ---
DATA_DIR = '.'
OUTPUT_DIR = '.'
EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_WORKERS = 2
NO_CUDA = False

# --- Device Setup ---
device = torch.device("cuda" if torch.cuda.is_available() and not NO_CUDA else "cpu")
logging.info(f"Using device: {device}")

# --- Data Loading ---
splits_path = os.path.join(DATA_DIR, 'dataset_splits.json')
with open(splits_path, 'r') as f:
    splits = json.load(f)

train_dataset = MalwareDataset(splits['train']['files'], splits['train']['labels'])
val_dataset = MalwareDataset(splits['validation']['files'], splits['validation']['labels'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
logging.info("Data loaders created.")

# --- Model Initialization ---
model = MalwareDetectionModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCELoss()
logging.info("Model initialized.")

# --- Training Loop ---
best_val_accuracy = 0.0
output_model_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
os.makedirs(OUTPUT_DIR, exist_ok=True)

logging.info("Starting training...")
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
    
    print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(model.state_dict(), output_model_path)
        logging.info(f"Validation accuracy improved. Saving model to {output_model_path}")

logging.info("Training finished.")
logging.info(f"Best validation accuracy: {best_val_accuracy:.4f}")

2025-11-02 21:00:41,087 - INFO - Using device: cuda
2025-11-02 21:00:41,421 - INFO - Data loaders created.
2025-11-02 21:00:54,426 - INFO - Model initialized.
2025-11-02 21:00:54,430 - INFO - Starting training...
